In [ ]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Gualberto"
STUDENT_ID = "TUPM-26-1130"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

In [ ]:
# [CELL 2] LOAD SECOM AND IDENTIFY USABLE FEATURES
WORK_DIR = Path.cwd()
DATA_DIR = next(
    (p for p in [WORK_DIR / "data", WORK_DIR.parent / "data"] if p.is_dir()),
    None
)
assert DATA_DIR is not None, "data/ folder not found (checked here and one level up)."
OUTPUT_DIR = WORK_DIR

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
outcomes = np.where(labels == -1, "Pass", "Fail")

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

print("Data Folder:", DATA_DIR.resolve())
print("Measurement Matrix:", raw_measurements.shape)
print("Pass Records:", (labels == -1).sum(), "| Fail Records:", (labels == 1).sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

In [ ]:
# [CELL 3] FEATURE SELECTION AND PREPARATION STRATEGIES
pass_mask = labels == -1
fail_mask = labels == 1

obs_pass = (~missing[pass_mask]).sum(axis=0)
obs_fail = (~missing[fail_mask]).sum(axis=0)

def select_ex3_features(seed):
    eligible = usable_indices[
        (obs_pass[usable_indices] >= 10) & (obs_fail[usable_indices] >= 10)
    ]
    rng = np.random.default_rng(seed)
    positions = rng.choice(len(eligible), size=5, replace=False)
    return eligible[positions], len(eligible)

ex3_indices, n_eligible = select_ex3_features(RANDOM_SEED)
ex3_features = feature_names[ex3_indices].tolist()

X3 = raw_measurements[:, ex3_indices]
full_medians3 = np.nanmedian(X3, axis=0)
X3_imputed = np.where(np.isnan(X3), full_medians3, X3)

STRATEGIES = ["Available-case", "Median-imputed"]

def group_values(strategy, j, mask):
    if strategy == "Available-case":
        col = X3[mask, j]
        return col[~np.isnan(col)]
    return X3_imputed[mask, j]

print("Eligible Features (>=10 observed per group):", n_eligible)
print("Selected Features:", ex3_features)
for strategy in STRATEGIES:
    diffs = [group_values(strategy, j, fail_mask).mean() - group_values(strategy, j, pass_mask).mean()
             for j in range(len(ex3_features))]
    print(f"{strategy} Mean Differences (Fail - Pass):", np.array(diffs))

In [ ]:
# [CELL 4] STRATIFIED BOOTSTRAP AND REPRODUCIBILITY CHECK
N_BOOT = 200 + 10 * SEED_DIGIT

def run_bootstrap(seed):
    rng = np.random.default_rng(seed)
    results = {}
    for strategy in STRATEGIES:
        for j, f in enumerate(ex3_features):
            pv = group_values(strategy, j, pass_mask)
            fv = group_values(strategy, j, fail_mask)
            p_idx = rng.integers(0, len(pv), size=(N_BOOT, len(pv)))
            f_idx = rng.integers(0, len(fv), size=(N_BOOT, len(fv)))
            results[(strategy, f)] = fv[f_idx].mean(axis=1) - pv[p_idx].mean(axis=1)
    return results

boot = run_bootstrap(RANDOM_SEED)
boot_again = run_bootstrap(RANDOM_SEED)

same_features, _ = select_ex3_features(RANDOM_SEED)
reproducible = bool(
    np.array_equal(same_features, ex3_indices)
    and all(np.array_equal(boot[k], boot_again[k]) for k in boot)
)

print("Bootstrap Repetitions:", N_BOOT)
print("Bootstrap Replicates Per Feature/Strategy:", len(next(iter(boot.values()))))
print("Reproducibility Verified:", reproducible)

In [ ]:
# [CELL 5] COMPARISON, EXPORT AND SENSITIVITY REPORT
rows = []
for strategy in STRATEGIES:
    for j, f in enumerate(ex3_features):
        pv = group_values(strategy, j, pass_mask)
        fv = group_values(strategy, j, fail_mask)
        diff = fv.mean() - pv.mean()
        d = boot[(strategy, f)]
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({
            "Feature": f, "Strategy": strategy,
            "Pass_N": len(pv), "Fail_N": len(fv),
            "Pass_Mean": pv.mean(), "Fail_Mean": fv.mean(),
            "Mean_Diff_Fail_minus_Pass": diff,
            "Direction": "Fail higher" if diff > 0 else ("Fail lower" if diff < 0 else "No difference"),
            "Bootstrap_Mean_Diff": d.mean(),
            "CI95_Lower": lo, "CI95_Upper": hi,
            "CI_Excludes_Zero": bool(lo > 0 or hi < 0),
        })

robust = pd.DataFrame(rows)
robust["Rank_by_Abs_Diff"] = (
    robust.groupby("Strategy")["Mean_Diff_Fail_minus_Pass"]
    .transform(lambda s: s.abs().rank(ascending=False, method="min"))
    .astype(int)
)

ac = robust[robust["Strategy"] == "Available-case"].set_index("Feature")
mi = robust[robust["Strategy"] == "Median-imputed"].set_index("Feature")

sign_changed = np.sign(ac["Mean_Diff_Fail_minus_Pass"]) != np.sign(mi["Mean_Diff_Fail_minus_Pass"])
rank_changed = ac["Rank_by_Abs_Diff"] != mi["Rank_by_Abs_Diff"]
ci_changed = ac["CI_Excludes_Zero"] != mi["CI_Excludes_Zero"]
change = (mi["Mean_Diff_Fail_minus_Pass"] - ac["Mean_Diff_Fail_minus_Pass"]).abs()

robust["Abs_Change_in_Mean_Diff"] = robust["Feature"].map(change)
robust["Sign_Changed"] = robust["Feature"].map(sign_changed)
robust["Rank_Changed"] = robust["Feature"].map(rank_changed)
robust["CI_Conclusion_Changed"] = robust["Feature"].map(ci_changed)

robustness_path = OUTPUT_DIR / "robustness_comparison.csv"
robust.to_csv(robustness_path, index=False)

largest_feature = change.idxmax()
largest_change = change.max()
sign_features = sign_changed[sign_changed].index.tolist()
rank_features = rank_changed[rank_changed].index.tolist()
ci_features = ci_changed[ci_changed].index.tolist()

lines = [
    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Selected Features: {', '.join(ex3_features)}",
    f"Bootstrap Repetitions: {N_BOOT}",
    f"Largest Change in Mean Difference: {largest_feature} ({largest_change:.4f})",
    f"Features with Changed Difference Sign: {', '.join(sign_features) if sign_features else 'None'}",
    f"Features with Changed Ranking: {', '.join(rank_features) if rank_features else 'None'}",
    f"Features with Changed CI Conclusion: {', '.join(ci_features) if ci_features else 'None'}",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Per-Feature Results (Fail minus Pass):",
]
for f in ex3_features:
    lines.append(
        f"- {f}: available-case {ac.loc[f, 'Mean_Diff_Fail_minus_Pass']:.4f} "
        f"[{ac.loc[f, 'CI95_Lower']:.4f}, {ac.loc[f, 'CI95_Upper']:.4f}] | "
        f"median-imputed {mi.loc[f, 'Mean_Diff_Fail_minus_Pass']:.4f} "
        f"[{mi.loc[f, 'CI95_Lower']:.4f}, {mi.loc[f, 'CI95_Upper']:.4f}]"
    )
lines += [
    "",
    "Interpretation Notes:",
    "- Both strategies describe the data; differences between them show sensitivity to missing-value handling.",
    "- Median imputation shrinks variability, so its intervals tend to be narrower than available-case intervals.",
    "- Rankings use raw absolute mean differences, which depend on feature scale.",
    "- A confidence interval that excludes zero indicates a group difference, not a cause of failure.",
]
report3 = "\n".join(lines) + "\n"

report3_path = OUTPUT_DIR / "sensitivity_report.txt"
report3_path.write_text(report3, encoding="utf-8")

print(robust.round(4))
print("--- Assessment Data: Exercise 3 ---")
print("Selected Features:", ex3_features)
print("Bootstrap Repetitions:", N_BOOT)
print("Largest Change in Mean Difference:", f"{largest_feature} ({largest_change:.4f})")
print("Features with Changed Difference Sign:", sign_features if sign_features else "None")
print("Features with Changed Ranking:", rank_features if rank_features else "None")
print("Reproducibility Verified:", reproducible)
print(robustness_path.name, "Created:", robustness_path.is_file())
print(report3_path.name, "Created:", report3_path.is_file())